# Test RAGAS

In [16]:
import sys
sys.path.insert(0, '/home/local/QCRI/fdeniz/projects/sspbench')

import importlib
import sspbench.novelty.ragas_utils
importlib.reload(sspbench.novelty.ragas_utils)

from sspbench.novelty.llm_utils import create_model_from_config
from sspbench.novelty.ragas_utils import (
    is_ragas_available,
    generate_qa_with_ragas,
    evaluate_qa_faithfulness,
    SentenceTransformerEmbeddings
)

print("✓ Imports successful")
print(f"RAGAS available: {is_ragas_available()}")

✓ Imports successful
RAGAS available: True


## Setup Eval Model

In [2]:
import os
# Set dummy OpenAI API key to prevent RAGAS from requiring real OpenAI credentials
os.environ["OPENAI_API_KEY"] = "dummy-key-for-ragas"

eval_config = {
    "type": "openai",
    "model": "gpt-oss",
    "api_url": "http://10.4.8.217:8000/v1",
    "api_token": "abc123",
    "api_version": "2024-12-01-preview"
}

try:
    eval_model = create_model_from_config(eval_config)
    print(f"✓ eval_model created successfully: {type(eval_model)}. Sample response: {eval_model.generate('Hello')}")
except Exception as e:
    print(f"✗ Failed to create eval_model: {e}")
    eval_model = None

embedding_model = SentenceTransformerEmbeddings("all-MiniLM-L6-v2")

Loading new model: gpt-oss
[Warning] Generation config not defined: {}, using defaults.
Discovered supported parameters: ['max_tokens', 'temperature', 'top_p', 'presence_penalty']
Generated Model: OpenaiLLM Fail on empty response: False
Content in batch was blocked by API model, trying chat inference...
✓ eval_model created successfully: <class 'models.openai_model.OpenaiLLM'>. Sample response: ['Hello! How can I assist you today?']


In [3]:
# Sample paragraph for testing
test_paragraph = """
The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. 
It is named after the engineer Gustave Eiffel, whose company designed and built the tower. 
Constructed from 1887 to 1889 as the entrance arch to the 1889 World's Fair, it was initially 
criticized by some of France's leading artists and intellectuals for its design, but it has 
become a global cultural icon of France and one of the most recognizable structures in the world.
""".strip()

print("Test paragraph:")
print(test_paragraph)
print("\n" + "="*80 + "\n")

Test paragraph:
The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. 
It is named after the engineer Gustave Eiffel, whose company designed and built the tower. 
Constructed from 1887 to 1889 as the entrance arch to the 1889 World's Fair, it was initially 
criticized by some of France's leading artists and intellectuals for its design, but it has 
become a global cultural icon of France and one of the most recognizable structures in the world.




In [4]:
# Configure RAGAS synthesizers for shorter answers
from ragas.testset.synthesizers import SingleHopSpecificQuerySynthesizer
from ragas.testset import TestsetGenerator
from sspbench.novelty.ragas_utils import CustomRagasLLM, CustomRagasEmbeddings

# Create RAGAS adapters
ragas_llm = CustomRagasLLM(eval_model, temperature=0.0)
ragas_embeddings = CustomRagasEmbeddings(embedding_model)

# Configure query distribution for ONLY short, specific questions
query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=ragas_llm), 1.0),  # 100% single-hop specific for shortest answers
]

print("✓ Configured query distribution for short answers:")
for synthesizer, weight in query_distribution:
    print(f"  {synthesizer.__class__.__name__}: {weight}")
print()

✓ Configured query distribution for short answers:
  SingleHopSpecificQuerySynthesizer: 1.0



In [5]:
if eval_model and is_ragas_available():
    print("Generating Q&A pairs with RAGAS using synthesizers...\n")
    try:
        # Create generator with synthesizer configuration
        generator = TestsetGenerator(
            llm=ragas_llm,
            embedding_model=ragas_embeddings,
        )

        from langchain_core.documents import Document
        doc = Document(page_content=test_paragraph)

        query_distribution = [
            (SingleHopSpecificQuerySynthesizer(llm=ragas_llm), 1.0),
        ]
        testset = generator.generate_with_langchain_docs(
            documents=[doc],
            testset_size=3,
            query_distribution=query_distribution
        )

        qa_pairs = []
        for idx, sample in enumerate(testset.samples, 1):
            qa_pairs.append(
                {
                    "id": str(idx),
                    "question": sample.eval_sample.user_input,
                    "answer": sample.eval_sample.reference,
                }
            )

        print(f"✓ Generated {len(qa_pairs)} Q&A pairs:\n")
        for i, qa in enumerate(qa_pairs, 1):
            print(f"Q{i}: {qa['question']}")
            print(f"A{i}: {qa['answer']}")
            print("-" * 80)
    except Exception as e:
        print(f"✗ Error generating Q&A: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠️ Skipping test - eval_model or RAGAS not available")

Generating Q&A pairs with RAGAS using synthesizers...



Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying ThemesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

...

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/3 [00:00<?, ?it/s]

✓ Generated 3 Q&A pairs:

Q1: As a cultural heritage tour guide, could you explain the significance of the year 1887 in the history of the Eiffel Tower, including how it relates to the tower's construction timeline, its purpose as the entrance arch to the 1889 World's Fair, and the initial reactions it received from artists and intellectuals?
A1: The year 1887 marks the beginning of the construction of the Eiffel Tower, a wrought‑iron lattice tower located on the Champ de Mars in Paris, France. Built by the company of engineer Gustave Eiffel, whose name the tower bears, the structure was erected from 1887 to 1889 to serve as the entrance arch for the 1889 World's Fair. Although the tower was initially criticized by some of France's leading artists and intellectuals for its design, it has since become a global cultural icon of France and one of the most recognizable structures in the world.
--------------------------------------------------------------------------------
Q2: Can you tel 

## Generate Q&A Pairs with Faithfulness Evaluation

In [6]:
# Evaluate each Q&A pair for faithfulness and relevancy
if 'qa_pairs' in locals() and qa_pairs:
    print("Evaluating Q&A pairs for faithfulness and relevancy...\n")
    for i, qa in enumerate(qa_pairs, 1):
        question = qa['question']
        answer = qa['answer']
        
        print(f"Evaluating Q&A pair {i}:")
        print(f"Question: {question}")
        print(f"Answer: {answer}")
        
        # Simple faithfulness evaluation using our custom model
        try:
            # Create a simple prompt to evaluate faithfulness
            eval_prompt = f"""
Given the question: "{question}"
And the answer: "{answer}"
And the context: "{test_paragraph}"

Rate the faithfulness of the answer on a scale of 0-1, where:
- 1.0 = The answer is completely faithful to the context
- 0.0 = The answer contradicts or is not supported by the context

Also rate the answer relevancy on a scale of 0-1, where:
- 1.0 = The answer directly and completely answers the question
- 0.0 = The answer does not address the question

Respond with just two numbers separated by a space, like: 0.95 0.88
"""

            eval_response = eval_model.generate(eval_prompt)
            # Parse the response
            parts = eval_response[0].strip().split()
            if len(parts) >= 2:
                faithfulness_score = float(parts[0])
                relevancy_score = float(parts[1])
            else:
                # Fallback if parsing fails
                faithfulness_score = 0.5
                relevancy_score = 0.5

            print(f"Faithfulness: {faithfulness_score:.4f}")
            print(f"Answer Relevancy: {relevancy_score:.4f}")

        except Exception as e:
            print(f"Error in custom evaluation: {e}")
            print("Using fallback scores: 0.5")
            print(f"Faithfulness: 0.50")
            print(f"Answer Relevancy: 0.50")
        
        print("-" * 80)
else:
    print("⚠️ No qa_pairs found. Please run the Q&A generation cell first.")

Evaluating Q&A pairs for faithfulness and relevancy...

Evaluating Q&A pair 1:
Question: As a cultural heritage tour guide, could you explain the significance of the year 1887 in the history of the Eiffel Tower, including how it relates to the tower's construction timeline, its purpose as the entrance arch to the 1889 World's Fair, and the initial reactions it received from artists and intellectuals?
Answer: The year 1887 marks the beginning of the construction of the Eiffel Tower, a wrought‑iron lattice tower located on the Champ de Mars in Paris, France. Built by the company of engineer Gustave Eiffel, whose name the tower bears, the structure was erected from 1887 to 1889 to serve as the entrance arch for the 1889 World's Fair. Although the tower was initially criticized by some of France's leading artists and intellectuals for its design, it has since become a global cultural icon of France and one of the most recognizable structures in the world.
Faithfulness: 1.0000
Answer Releva

In [17]:
# Evaluate Q&A pairs using RAGAS evaluate_qa_faithfulness function
if 'qa_pairs' in locals() and qa_pairs and eval_model is not None:
    print("Evaluating Q&A pairs using RAGAS evaluation function...\n")
    
    from sspbench.novelty.ragas_utils import evaluate_qa_faithfulness
    
    for i, qa in enumerate(qa_pairs, 1):
        question = qa['question']
        answer = qa['answer']
        
        print(f"Evaluating Q&A pair {i} (RAGAS):")
        print(f"Question: {question}")
        print(f"Answer: {answer}")
        
        try:
            # Use RAGAS evaluation function
            metrics = evaluate_qa_faithfulness(
                question=question,
                answer=answer,
                context=test_paragraph,
                eval_model=ragas_llm,
                embedding_model=ragas_embeddings
            )
            
            print(f"Faithfulness: {metrics['faithfulness']:.4f}")
            print(f"Answer Relevancy: {metrics['answer_relevancy']:.4f}")
            
        except Exception as e:
            print(f"Error in RAGAS evaluation: {e}")
            print("Using fallback scores: 0.5")
            print(f"Faithfulness: 0.50")
            print(f"Answer Relevancy: 0.50")
        
        print("-" * 80)
else:
    print("⚠️ No qa_pairs found or eval_model not available. Please run the Q&A generation cell first.")

Evaluating Q&A pairs using RAGAS evaluation function...

Evaluating Q&A pair 1 (RAGAS):
Question: As a cultural heritage tour guide, could you explain the significance of the year 1887 in the history of the Eiffel Tower, including how it relates to the tower's construction timeline, its purpose as the entrance arch to the 1889 World's Fair, and the initial reactions it received from artists and intellectuals?
Answer: The year 1887 marks the beginning of the construction of the Eiffel Tower, a wrought‑iron lattice tower located on the Champ de Mars in Paris, France. Built by the company of engineer Gustave Eiffel, whose name the tower bears, the structure was erected from 1887 to 1889 to serve as the entrance arch for the 1889 World's Fair. Although the tower was initially criticized by some of France's leading artists and intellectuals for its design, it has since become a global cultural icon of France and one of the most recognizable structures in the world.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
/home/local/QCRI/fdeniz/anaconda3/envs/autobencher/lib/python3.10/queue.py:165: RuntimeWarning: coroutine 'CustomRagasLLM.generate' was never awaited
  with self.not_empty:


Faithfulness: 1.0000
Answer Relevancy: 0.7198
--------------------------------------------------------------------------------
Evaluating Q&A pair 2 (RAGAS):
Question: Can you tel me about the histroy and architecure of the Eiffel Towr in Pariss, includng why it was built, who designd it, and how people first reactd to it?
Answer: The Eiffel Tower is a wrought‑iron lattice tower located on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower. It was constructed from 1887 to 1889 as the entrance arch to the 1889 World's Fair. Although some of France's leading artists and intellectuals initially criticized its design, it has become a global cultural icon of France and one of the most recognizable structures in the world.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Faithfulness: 1.0000
Answer Relevancy: 0.6154
--------------------------------------------------------------------------------
Evaluating Q&A pair 3 (RAGAS):
Question: Who be Gustave Eiffel and what he did for the Eiffel Tower?
Answer: Gustave Eiffel was the engineer whose company designed and built the Eiffel Tower, and the tower is named after him.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Faithfulness: 1.0000
Answer Relevancy: 0.7714
--------------------------------------------------------------------------------


## Comparison: Direct Evaluation vs RAGAS Evaluation

This cell compares the results from:
- **Cell 8**: Direct evaluation using custom prompts with your model
- **Cell 9**: RAGAS evaluation using built-in faithfulness and answer relevancy metrics

Both methods use your custom model but different evaluation approaches.

## Summary

This notebook demonstrates:
1. ✓ RAGAS setup with SingleHopSpecificQuerySynthesizer
2. ✓ Q&A pair generation with faithfulness evaluation for each pair
3. ✓ Comparison between direct model evaluation and RAGAS evaluation metrics
4. ✓ Clean, focused testing of the RAGAS integration

Each generated Q&A pair includes faithfulness and answer relevancy scores from both evaluation methods.